In [3]:
import sqlite3

conn = sqlite3.connect("students.db")
cursor = conn.cursor()

print("Connected to students.db")
print(type(conn), type(cursor))

Connected to students.db
<class 'sqlite3.Connection'> <class 'sqlite3.Cursor'>


In [5]:
# 1 - Students in highest year level
cursor.execute("SELECT full_name, course FROM students WHERE year_level = (SELECT MAX(year_level) FROM students)")
print(cursor.fetchall())


[('Anna Reyes', 'BS ECE')]


In [6]:
# 2 - Students in any Engineering course
cursor.execute("SELECT * FROM students WHERE course IN (SELECT course_code FROM courses WHERE description LIKE '%Engineering%')")
print(cursor.fetchall())

[(1, 'Anna Reyes', 'BS ECE', 4), (2, 'Mika Santos', 'BS CE', 2)]


In [11]:
# 3 - Courses with more than 1 student
cursor.execute("SELECT course, COUNT(*) AS total FROM students GROUP BY course HAVING total > 1")
print(cursor.fetchall())


[]


In [9]:
# 4 - Graduating status
cursor.execute("SELECT *, CASE WHEN year_level >= 4 THEN 'Graduating' ELSE 'Regular' END AS status FROM students")
print(cursor.fetchall())

[(1, 'Anna Reyes', 'BS ECE', 4, 'Graduating'), (2, 'Mika Santos', 'BS CE', 2, 'Regular')]


In [12]:
# 5 - LEFT JOIN (all students even without course match)
cursor.execute("SELECT s.full_name, c.description FROM students s LEFT JOIN courses c ON s.course = c.course_code")
print(cursor.fetchall())

[('Anna Reyes', 'Electronics and Communications Engineering'), ('Mika Santos', 'Civil Engineering'), ('Anna Reyes', 'Electronics and Communications Engineering'), ('Carlos Dela Cruz', 'Electrical Engineering'), ('Mika Santos', 'Civil Engineering'), ('Anna Reyes', 'Electronics and Communications Engineering'), ('Carlos Dela Cruz', 'Electrical Engineering'), ('Mika Santos', 'Civil Engineering')]


In [14]:
# 6 - Students with 3-part names
cursor.execute("SELECT COUNT(*) FROM students WHERE full_name LIKE '% % %'")
print(cursor.fetchone())

(2,)


In [15]:
# 7 - Highest year level per course
cursor.execute("SELECT course, MAX(year_level) FROM students GROUP BY course ORDER BY MAX(year_level) DESC")
print(cursor.fetchall())

[('BS ECE', 4), ('BS EE', 2), ('BS CE', 2)]


In [16]:
# 8 - Even-numbered rowids
cursor.execute("SELECT * FROM students WHERE rowid % 2 = 0")
print(cursor.fetchall())

[(2, 'Mika Santos', 'BS CE', 2), (4, 'Carlos Dela Cruz', 'BS EE', 2), (6, 'Anna Reyes', 'BS ECE', 3), (8, 'Mika Santos', 'BS CE', 1)]


In [17]:
# 9 - Names with at least two e's
cursor.execute("SELECT full_name FROM students WHERE full_name GLOB '*e*e*'")
print(cursor.fetchall())

[('Anna Reyes',), ('Anna Reyes',), ('Anna Reyes',)]


In [18]:
# 10 - Students with no matching course record
cursor.execute("SELECT full_name, course FROM students WHERE NOT EXISTS (SELECT 1 FROM courses WHERE course_code = students.course)")
print(cursor.fetchall())

[]


In [19]:
# 11 - Course stats
cursor.execute("SELECT course, COUNT(*), AVG(year_level) FROM students GROUP BY course")
print(cursor.fetchall())


[('BS CE', 3, 1.3333333333333333), ('BS ECE', 3, 3.3333333333333335), ('BS EE', 2, 2.0)]


In [20]:
# 12 - Initial and full name
cursor.execute("SELECT full_name, SUBSTR(full_name, 1, 1) AS initial FROM students")
print(cursor.fetchall())

[('Anna Reyes', 'A'), ('Mika Santos', 'M'), ('Anna Reyes', 'A'), ('Carlos Dela Cruz', 'C'), ('Mika Santos', 'M'), ('Anna Reyes', 'A'), ('Carlos Dela Cruz', 'C'), ('Mika Santos', 'M')]


In [22]:
# 13 - Dynamic search with LIMIT 1
cursor.execute("SELECT * FROM students WHERE full_name LIKE ? LIMIT 1", ('%' + input("Enter keyword: ") + '%',))
print(cursor.fetchone())


None


In [23]:
# 14 - Most popular course
cursor.execute("SELECT course, COUNT(*) FROM students GROUP BY course ORDER BY COUNT(*) DESC LIMIT 1")
print(cursor.fetchall())

[('BS ECE', 3)]


In [24]:
# 15 - Unique courses of students with 'a' in name, year 2-4
cursor.execute("SELECT DISTINCT course FROM students WHERE full_name LIKE '%a%' AND year_level BETWEEN 2 AND 4")
print(cursor.fetchall())

[('BS ECE',), ('BS CE',), ('BS EE',)]


In [25]:
# 16 - Student with longest name
cursor.execute("SELECT * FROM students WHERE LENGTH(full_name) = (SELECT MAX(LENGTH(full_name)) FROM students)")
print(cursor.fetchall())

[(4, 'Carlos Dela Cruz', 'BS EE', 2), (7, 'Carlos Dela Cruz', 'BS EE', 2)]


In [26]:
# 17 - Count BS CE students in lowest year level
cursor.execute("SELECT COUNT(*) FROM students WHERE course = 'BS CE' AND year_level = (SELECT MIN(year_level) FROM students WHERE course = 'BS CE')")
print(cursor.fetchall())

[(2,)]


In [27]:
# 18 - Students in courses with "Computer" in description
cursor.execute("SELECT s.full_name, s.year_level, c.description FROM students s JOIN courses c ON s.course = c.course_code WHERE c.description LIKE '%Computer%'")
print(cursor.fetchall())

[]


In [28]:
# 19 - Error handling
try:
    cursor.execute("SELECT * FROM unknown_table")
except Exception as e:
    print("Error:", e)


Error: no such table: unknown_table


In [29]:
# 20 - Course(s) with most students (handles ties)
cursor.execute("SELECT course, COUNT(*) FROM students GROUP BY course HAVING COUNT(*) = (SELECT MAX(cnt) FROM (SELECT COUNT(*) AS cnt FROM students GROUP BY course))")
print(cursor.fetchall())

[('BS CE', 3), ('BS ECE', 3)]
